# 05. Σ (総和), Π (総乗), ∫ (積分)

**このノートブックの内容**:
- $\sum$ (Sigma): 足し算の繰り返し → `sum()` / `np.sum`
- $\prod$ (Pi): 掛け算の繰り返し → `math.prod()` / `np.prod`
- $\int$ (Integral): 連続値の和 → `scipy.integrate.quad`
- 順列・組合せ ($n!$, $\binom{n}{k}$)

> 🧭 **クイックナビ**: 📚 [ROOT (全体 TOP)](../../README.md) ・ 🏠 [章 TOP](../README.md) ・ 📖 [解説 md (05_summation_product.md)](../05_summation_product.md)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore", message=".*distutils Version classes.*", category=DeprecationWarning)
import japanize_matplotlib  # noqa: F401  # 日本語フォント (豆腐化対策)

%matplotlib inline

## 1. $\sum_{i=1}^{n} a_i$ — 総和

$$\sum_{i=1}^{5} i = 1 + 2 + 3 + 4 + 5 = 15$$

In [ ]:
# Python 組込み sum()
total = sum(range(1, 6))
print(f'Σ_{{i=1}}^5 i = {total}')

# NumPy 版 (大規模配列向け、JIT 連携も可能)
x = np.arange(1, 6)
print(f'np.sum: {np.sum(x)}')

# 重み付き総和 (機械学習で頻出: 加重平均)
weights = np.array([0.1, 0.2, 0.3, 0.2, 0.2])
values  = np.array([10, 20, 30, 40, 50])
weighted = np.sum(weights * values)
print(f'\n加重平均 = Σ w_i x_i = {weighted}')

## 2. $\sum_{i=1}^{n} i^2$ など — 関数の和

ジェネレータ式 `sum(f(i) for i in range(...))` が便利。

In [ ]:
# Σ_{i=1}^{10} i^2
sum_squares = sum(i**2 for i in range(1, 11))
print(f'Σ i² (i=1..10) = {sum_squares}')

# 公式 n(n+1)(2n+1)/6 と一致するか検算
n = 10
formula = n * (n + 1) * (2*n + 1) // 6
print(f'公式 n(n+1)(2n+1)/6 = {formula}')

## 3. $\prod_{i=1}^{n} a_i$ — 総乗

$$\prod_{i=1}^{5} i = 1 \cdot 2 \cdot 3 \cdot 4 \cdot 5 = 120 = 5!$$

In [ ]:
# math.prod (Python 3.8+)
product = math.prod(range(1, 6))
print(f'Π_{{i=1}}^5 i = {product}  ← これは 5! と同じ')
print(f'math.factorial(5) = {math.factorial(5)}')

# NumPy 版
print(f'np.prod(arange(1,6)): {np.prod(np.arange(1, 6))}')

## 4. $n!$ (階乗) と $\binom{n}{k}$ (二項係数)

組合せ論で頻出。

In [ ]:
from math import factorial, comb

print(f'5! = {factorial(5)}')
print(f'10! = {factorial(10):,}')

# 二項係数 C(n, k) = n! / (k! (n-k)!)
print(f'\nC(5, 2) = {comb(5, 2)}  (5 個から 2 個を選ぶ組合せ数)')
print(f'C(10, 3) = {comb(10, 3)}')

## 5. $\int$ — 連続値の和 (積分)

離散の $\sum$ を「連続」 にした概念。SciPy の `quad` で数値積分できる。

例: $\int_0^1 x^2 \, dx = \frac{1}{3}$

In [ ]:
from scipy.integrate import quad

result, error = quad(lambda x: x**2, 0, 1)
print(f'∫_0^1 x² dx = {result:.6f}  (誤差 ±{error:.1e})')
print(f'解析解 1/3 = {1/3:.6f}')

# 円の面積 = π を積分で求める: ∫_{-1}^1 2√(1-x²) dx
result, _ = quad(lambda x: 2 * np.sqrt(1 - x**2), -1, 1)
print(f'\n単位円の面積 ≈ {result:.6f}  (π ≈ {math.pi:.6f})')

## 5-2. $\Gamma$ (ガンマ関数) — 階乗を実数へつなぐ

$n!$ は整数の上にしか点がありません。その点を**なめらかにつないだ曲線**が $\Gamma$ です。

$$
\Gamma(z) = \int_0^{\infty} t^{\,z-1}e^{-t}\,dt, \qquad \Gamma(z+1) = z\,\Gamma(z)
$$

⚠️ **1 つズレます**: $\Gamma(n) = (n-1)!$ であって $n!$ ではありません。

In [ ]:
from scipy.special import gamma, gammaln, beta as beta_fn

# --- Γ(n) = (n−1)! の確認 ---
for n in range(1, 6):
    print(f'Γ({n}) = {gamma(n):>4.0f}   ({n-1}! = {math.factorial(n-1)})')

print(f'\nΓ(0.5) = {gamma(0.5):.4f}   (√π = {np.sqrt(np.pi):.4f})  ← 整数以外でも値がある')

# --- 階乗の点と Γ 曲線を重ねて描く ---
z = np.linspace(0.1, 5.5, 400)          # shape: (400,)
n_int = np.arange(1, 6)                  # 1..5

plt.figure(figsize=(8, 4))
plt.plot(z, gamma(z), lw=2, label=r'$\Gamma(z)$ (連続)')
plt.scatter(n_int, [math.factorial(n - 1) for n in n_int], s=80, color='C3',
            zorder=3, label=r'$(n-1)!$ (整数の点)')
plt.ylim(0, 30); plt.xlabel('z'); plt.ylabel(r'$\Gamma(z)$')
plt.title('ガンマ関数は「階乗の点」を通るなめらかな曲線')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# --- ⚠️ すぐオーバーフローする -> 対数版 gammaln ---
print(f'Γ(171) = {gamma(171):.3e}   ← ここまでは足りる')
print(f'Γ(172) = {gamma(172)}   ← float64 (最大 ~1.8e308) を超える')
print(f'log Γ(172)   = {gammaln(172):.4f}')
print(f'log Γ(10000) = {gammaln(10000):.1f}   ← 桁がいくら大きくても平気')

### どこで出会うか — ほとんどは「正規化定数」として

$\Gamma$ を単体で使うことは少なく、**確率密度の頭に付く係数**として現れます。

$$
B(\alpha,\beta) = \frac{\Gamma(\alpha)\Gamma(\beta)}{\Gamma(\alpha+\beta)},
\qquad
f_{\mathrm{Beta}}(x) = \frac{\Gamma(\alpha+\beta)}{\Gamma(\alpha)\Gamma(\beta)}x^{\alpha-1}(1-x)^{\beta-1}
$$

この「$\Gamma$ の塊」は**面積を 1 にするための割り算**で、ベイズで $\propto$ と書いて省略される部分そのものです。

In [ ]:
from scipy import stats

ALPHA: float = 2.0
BETA: float = 5.0
x_pt: float = 0.3

manual = x_pt**(ALPHA - 1) * (1 - x_pt)**(BETA - 1) / beta_fn(ALPHA, BETA)
via_gamma = (gamma(ALPHA + BETA) / (gamma(ALPHA) * gamma(BETA))
             * x_pt**(ALPHA - 1) * (1 - x_pt)**(BETA - 1))

print(f'手計算       : {manual:.6f}')
print(f'Γ で書いた版 : {via_gamma:.6f}')
print(f'SciPy        : {stats.beta.pdf(x_pt, ALPHA, BETA):.6f}')
assert np.isclose(manual, stats.beta.pdf(x_pt, ALPHA, BETA))
print('✅ 3 つとも一致 -> Γ の塊は「正規化定数」')

# 正規化定数を外すと面積が 1 にならない (∝ で省略されるのはこの部分)
xs = np.linspace(1e-6, 1 - 1e-6, 2000)
unnorm = xs**(ALPHA - 1) * (1 - xs)**(BETA - 1)      # Γ の係数なし
print(f'\n正規化なしの面積 : {np.trapezoid(unnorm, xs):.6f}  (= B(α,β) = {beta_fn(ALPHA, BETA):.6f})')
print(f'正規化ありの面積 : {np.trapezoid(unnorm / beta_fn(ALPHA, BETA), xs):.6f}')

## 6. 機械学習に効いてくる Σ — 損失関数の平均

ML の損失関数は「全データの誤差を足して平均する」 のがほぼお決まり:

$$L = \frac{1}{N} \sum_{i=1}^{N} (y_i - \hat{y}_i)^2$$

(平均二乗誤差 MSE)

In [ ]:
# ダミーデータ
y_true = np.array([3.0, 5.0, 7.0, 9.0])
y_pred = np.array([2.8, 5.1, 7.3, 8.9])

# MSE を素朴に書く
N = len(y_true)
mse_loop = sum((y_true[i] - y_pred[i])**2 for i in range(N)) / N
print(f'MSE (ループ版): {mse_loop:.4f}')

# NumPy のベクトル演算で 1 行
mse_np = np.mean((y_true - y_pred)**2)
print(f'MSE (NumPy): {mse_np:.4f}')

## ⚠️ よくあるエラー — 0 で割る (数学とプログラミングの差)

数学的に $1/0$ は未定義。Python と NumPy では **扱いが違います**。両方を体感しておきましょう。

In [ ]:
import numpy as np
import warnings

# 1) Python 組込み: ZeroDivisionError
try:
    x = 1 / 0
except ZeroDivisionError as e:
    print(f'(Python) 1 / 0 → ❌ {type(e).__name__}: {e}')

# 2) NumPy: 警告 + 無限大 (例外にはならない)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')   # ここでは警告抑制して結果だけ見る
    print(f'(NumPy)  [1.0] / [0.0] = {np.array([1.0]) / np.array([0.0])}  ← inf')
    print(f'(NumPy)  [0.0] / [0.0] = {np.array([0.0]) / np.array([0.0])}  ← nan')

print('  → 数学的に未定義のものは、Python は例外、NumPy は inf/nan で表現する')
# エラー読解の詳細: start_here/00_pet_terminal/columns/05_reading_errors.md

---

## 📍 ナビゲーション

| ← 前 | 🏠 章 TOP | 📚 全体 TOP | 次 → |
|---|---|---|---|
| [04_function_notation.ipynb](04_function_notation.ipynb) | [章 TOP](../README.md) | [📚 ROOT README](../../README.md) | [06_greek_letters.ipynb](06_greek_letters.ipynb) |